# ORA lollipop plots — WGCNA modules (Clinical Outcome)

Visualises the statistically significant over-represented terms from the ORA
(`gseapy.enrich`) CSVs written by `ora.ipynb` into
`outputs/images/clinical_outcome/wgcna/ora/{MODULE}/{hub|comp_set}/`.

**Design.** ORA has no direction/NES (every reported term is *over*-represented), so the
GSEA diverging dot plot does not apply. Each database gets a horizontal **lollipop**: stem
length (**x**) = `-log10(Adjusted P-value)` (FDR), dot **size** = matched gene count, dot
**colour** = **Odds Ratio** on the `rocket` ramp (brighter = stronger enrichment).

*Why Odds Ratio and not Gene Ratio:* Gene Ratio (`matched / set_size`) has a denominator
that changes with every pathway, so its values are **not comparable across terms** — Odds
Ratio is comparable and is independent of the x-axis (which already carries FDR). The size
legend sits **outside** the axes so it never overlaps the dots.

**Compact by design.** `TOP_N = 5` terms per figure, drawn on a tight `ROW_HEIGHT_IN` pitch
with the y-limits pulled in to the rows themselves, in 9 pt Arial Narrow — no wide gaps
between successive terms.

**One control:** set `MODULE` below to `"turquoise"` or `"blue"` and run. The notebook
loops internally over both gene lists (`hub`, `comp_set`) and every database present, and
writes one SVG per database — the top `TOP_N` terms (by FDR).

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
import seaborn as sns

## Configuration

`MODULE` is the only thing to change.

In [ ]:
# ---- pick ONE module here ---------------------------------------------------
MODULE = "blue"          # "turquoise" | "blue"

BASE_DIR = Path(f"{ACC_DATA_ROOT}/bulk_rna_data/outputs/images/clinical_outcome/wgcna")
ORA_DIR  = BASE_DIR / "ora" / MODULE

FDR_CUTOFF = 0.05        # significance cut on Adjusted P-value
TOP_N      = 5           # terms per figure

GENE_LISTS = ["hub", "comp_set"]

# CSV basename -> short database tag used in figure filenames / titles
DB_SHORT = {
    "MSigDB_C5_BP.gmt.csv":       "BP",
    "MSigDB_C5_CC.gmt.csv":       "CC",
    "MSigDB_C5_MF.gmt.csv":       "MF",
    "MSigDB_C2_Reactome.gmt.csv": "reactome",
    "MSigDB_Hallmark.gmt.csv":    "hallmark",
}

# prefixes stripped from term names for display
DB_PREFIXES = ('GOBP_', 'GOCC_', 'GOMF_', 'REACTOME_', 'HALLMARK_')
LABEL_MAXLEN = 55        # truncate very long term labels

# ---- compact geometry -------------------------------------------------------
# Row pitch in inches. The axes height is ROW_HEIGHT_IN * n, and the y-limits are clamped to
# the rows (see `lollipop_plot`), so this IS the gap between successive terms — matplotlib's
# default categorical padding is what otherwise strands 5 terms across a tall axes.
ROW_HEIGHT_IN = 0.22
FIG_W_IN      = 6.5      # 9 pt Arial Narrow labels need far less width than the old 14 pt
# Dot areas. The max is capped so the biggest dot (diameter ~14.7 pt) still clears the
# 0.22 in (15.8 pt) row pitch above; the min is kept well off zero so the smallest matched
# count is still a readable dot rather than a speck.
DOT_S_MIN, DOT_S_MAX = 30, 170

# Dot colour ramp — sequential, perceptually uniform 'rocket' (dark purple -> orange).
# Deliberately NOT 'mako' (the GSEA dot plot's ramp) and NOT the green/red NES colours, so
# this figure stays visually distinct from the GSEA plot. It encodes the Odds Ratio — an
# enrichment-strength axis that is comparable across terms and independent of the x-axis
# (which carries -log10 FDR). Gene Ratio is deliberately NOT used: its per-term denominator
# (set size) varies, so gene-ratio values are not comparable between pathways.
FILL_CMAP = sns.color_palette('rocket', as_cmap=True)

# ---- fonts ------------------------------------------------------------------
# Arial Narrow is NOT installed here. Liberation Sans Narrow is, and is metrically identical
# to Arial Narrow (same advance widths). Listing both means:
#   * matplotlib resolves metrics from LiberationSansNarrow-Regular.ttf -> Arial-Narrow-correct
#     layout
#   * svg.fonttype='none' writes the stack "'Arial Narrow', 'Liberation Sans Narrow'" into the
#     SVG, so Illustrator/journals with Arial Narrow render true Arial Narrow, others fall
#     back cleanly.
# Setting font.family='Arial Narrow' alone would silently lay out with DejaVu Sans, which is
# not narrow at all — every label would be laid out far too wide.
plt.rcParams['font.family'] = ['Arial Narrow', 'Liberation Sans Narrow']
plt.rcParams['svg.fonttype'] = 'none'      # keep SVG text editable in Illustrator/Inkscape

# 'Arial Narrow' is deliberately first so it lands in the SVG's font-family, but since it is
# not installed here font_manager logs "findfont: Font family 'Arial Narrow' not found" for
# EVERY text object. The lookup then falls through to Liberation Sans Narrow, which is what we
# want, so the message is noise. It is a logging call, not a warning, so filterwarnings does
# not catch it.
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

# ---- font sizes -------------------------------------------------------------
# 9 pt throughout, titles included — a larger title would reintroduce vertical space above a
# figure whose whole point is to be compact.
FONT_PT = 9
plt.rcParams['font.size']             = FONT_PT
plt.rcParams['axes.titlesize']        = FONT_PT
plt.rcParams['axes.labelsize']        = FONT_PT
plt.rcParams['xtick.labelsize']       = FONT_PT
plt.rcParams['ytick.labelsize']       = FONT_PT
plt.rcParams['legend.fontsize']       = FONT_PT
plt.rcParams['legend.title_fontsize'] = FONT_PT

print(f'module   : {MODULE}')
print(f'ora dir  : {ORA_DIR}')
print(f'font     : {plt.rcParams["font.family"]}')

## Helpers

In [ ]:
def clean_term(term, maxlen=LABEL_MAXLEN):
    """Strip the MSigDB database prefix, de-underscore and tidy a term for display."""
    label = str(term)
    for prefix in DB_PREFIXES:
        if label.startswith(prefix):
            label = label[len(prefix):]
            break
    label = label.replace('_', ' ').strip().title()
    if maxlen is not None and len(label) > maxlen:
        label = label[:maxlen - 1].rstrip() + '…'
    return label


def parse_overlap(s):
    """'167/301' -> (matched=167, set_size=301). gseapy enrich Overlap = matched/geneset."""
    matched, set_size = str(s).split('/')
    return int(matched), int(set_size)


def neglog10_fdr(fdr_series):
    """-log10(FDR) with exact/underflowed zeros floored at half the smallest non-zero FDR
    in this file. The resulting y-ceiling is then an artifact of the floor, not a real
    infinity - do not over-read the top of the axis."""
    fdr = fdr_series.astype(float).copy()
    nonzero = fdr[fdr > 0]
    floor = (nonzero.min() / 2.0) if len(nonzero) else 1e-300
    fdr = fdr.clip(lower=floor)
    return -np.log10(fdr)


def select_top(sig, n):
    """Top n terms: Adjusted P-value ascending, ties broken by Odds Ratio descending.

    Many terms tie at FDR ~ 0, so ranking on FDR alone would return an arbitrary slice of
    the tie block. The Odds-Ratio tie-break surfaces the strongest enrichments within it
    (same rationale as the GSEA fdr+|NES| tie-break).
    """
    return (sig.sort_values(['Adjusted P-value', 'Odds Ratio'], ascending=[True, False])
               .head(n))


def load_sig(csv_path):
    """Read one ORA CSV, derive plotting columns, keep only FDR < FDR_CUTOFF.

    `matched` = numerator of the Overlap "167/301" string (dot size). `Odds Ratio` is read
    straight from the gseapy CSV (dot colour). Gene Ratio is intentionally not derived — its
    per-term denominator (set size) varies, so the values aren't comparable across pathways.
    """
    df = pd.read_csv(csv_path)
    df[['matched', 'set_size']] = df['Overlap'].apply(
        lambda s: pd.Series(parse_overlap(s)))
    df['neglog10'] = neglog10_fdr(df['Adjusted P-value'])
    df['label'] = df['Term'].apply(clean_term)
    return df[df['Adjusted P-value'] < FDR_CUTOFF].copy()


def save_fig(fig, path):
    """Save a figure as SVG (600 dpi)."""
    fig.savefig(path, format='svg', dpi=600, bbox_inches='tight')
    print(f'[saved] {path}')

## Lollipop

`-log10(FDR)` on x (most significant terms at the top), dot **size** = matched genes, dot
**colour** = **Odds Ratio** on the `rocket` ramp (brighter = stronger enrichment). The size
legend and colorbar both sit **outside** the axes, so no element overlaps the dots. Term
labels can repeat across databases but are unique within one figure.

In [ ]:
def lollipop_plot(df, title, out_path):
    """Draw and save one ORA lollipop for an already-filtered/selected set of terms.

    x = -log10(FDR); dot size = matched gene count; dot colour = Odds Ratio on the rocket
    ramp. Size legend and colorbar are drawn OUTSIDE the axes so nothing overlaps the dots.
    Most significant term at the top. Rows sit on a fixed ROW_HEIGHT_IN pitch.
    """
    if df is None or len(df) == 0:
        print(f'[skip] nothing to plot for {out_path.name}')
        return

    # most significant at the TOP of a matplotlib y-axis -> ascending order
    top = df.sort_values('neglog10', ascending=True).copy()
    n = len(top)

    # matched gene count -> marker area, capped so a dot stays inside its row
    ms = top['matched'].astype(float)
    if ms.max() > ms.min():
        sizes = DOT_S_MIN + (ms - ms.min()) / (ms.max() - ms.min()) * (DOT_S_MAX - DOT_S_MIN)
    else:
        sizes = pd.Series((DOT_S_MIN + DOT_S_MAX) / 2.0, index=ms.index)

    # dot colour = Odds Ratio; per-figure min->max norm for dynamic range
    odds = top['Odds Ratio'].astype(float)
    if float(odds.max()) > float(odds.min()):
        norm = Normalize(vmin=float(odds.min()), vmax=float(odds.max()))
    else:
        norm = Normalize(vmin=0, vmax=float(odds.max()) if odds.max() > 0 else 1.0)

    # height is the row pitch times the row count, plus a fixed allowance for the title,
    # x-axis and tick labels — so adding terms lengthens the figure instead of stretching
    # the gaps between the ones already there
    fig, ax = plt.subplots(figsize=(FIG_W_IN, ROW_HEIGHT_IN * n + 1.0))

    ax.hlines(y=top['label'], xmin=0, xmax=top['neglog10'],
              color='0.7', linewidth=1.0, zorder=1)
    ax.scatter(top['neglog10'], top['label'],
               s=sizes, c=odds, cmap=FILL_CMAP, norm=norm,
               edgecolor='white', linewidth=0.6, zorder=3)

    ax.set_xlim(left=0)
    # clamp the y-limits to the rows themselves; the matplotlib default leaves half a row of
    # padding top and bottom, which on a 5-row figure is 20% wasted height
    ax.set_ylim(-0.6, n - 0.4)
    ax.grid(axis='x', color='0.9', linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    ax.set_xlabel('−log10(FDR)')   # plain text (no mathtext) so it stays in the Arial stack
    ax.set_ylabel('')
    ax.set_title(title, pad=6)
    ax.tick_params(axis='both', length=3, pad=2)

    cbar = fig.colorbar(ScalarMappable(norm=norm, cmap=FILL_CMAP), ax=ax, pad=0.02,
                        shrink=0.9, aspect=12)
    cbar.set_label('Odds Ratio')   # brighter = stronger enrichment; not inverted
    cbar.ax.tick_params(length=2, pad=2)

    # size legend at evenly spaced representative matched-gene counts, drawn OUTSIDE the
    # axes (upper-left of the right margin) so it never overlaps the dots
    ticks = np.unique(np.linspace(ms.min(), ms.max(), 3).round().astype(int))
    size_handles = []
    for t in ticks:
        if ms.max() > ms.min():
            sz = DOT_S_MIN + (t - ms.min()) / (ms.max() - ms.min()) * (DOT_S_MAX - DOT_S_MIN)
        else:
            sz = (DOT_S_MIN + DOT_S_MAX) / 2.0
        size_handles.append(ax.scatter([], [], s=sz, c='0.55', edgecolor='white',
                                       linewidth=0.6, label=str(int(t))))
    ax.legend(handles=size_handles, title='Matched genes',
              loc='upper left', bbox_to_anchor=(1.22, 1.0), frameon=False,
              labelspacing=1.0, borderpad=0.4, handletextpad=0.9)

    sns.despine(ax=ax, left=True)
    fig.tight_layout()
    save_fig(fig, out_path)
    plt.close(fig)

## Driver

For each gene list and every database CSV present, write one lollipop figure of the top
`TOP_N` terms (ranked by FDR, tie-broken by Odds Ratio). Databases with fewer than `TOP_N`
significant terms show all of them (no padding).

In [ ]:
for gene_list in GENE_LISTS:
    csv_dir = ORA_DIR / gene_list
    if not csv_dir.is_dir():
        print(f'[skip] no directory {csv_dir}')
        continue

    for csv_path in sorted(csv_dir.glob('*.gmt.csv')):
        db = DB_SHORT.get(csv_path.name, csv_path.name.replace('.gmt.csv', ''))
        sig = load_sig(csv_path)
        print(f'{MODULE}/{gene_list}/{db}: {len(sig)} significant terms (FDR < {FDR_CUTOFF})')
        if len(sig) == 0:
            continue

        # top-N figure
        top = select_top(sig, TOP_N)
        lollipop_plot(
            top,
            f'{MODULE} {gene_list} — {db} (top {len(top)}, FDR < {FDR_CUTOFF})',
            csv_dir / f'ora_lollipop_{gene_list}_{db}_top{TOP_N}.svg',
        )

print('done.')